In [0]:
%sql

SELECT * FROM ecomotive.landing.bronze_sensors 
ORDER BY timestamp, id_camion;

In [0]:
# --- Batch 04: DATOS TÓXICOS ---
# Ruta de inbox (la misma de siempre)
path_inbox = "/Volumes/ecomotive/landing/ecomotive_vol/inbox"

json_batch_toxic = """
{"id_camion": 105, "velocidad": 80, "rpm": 2000, "timestamp": "2023-01-01T13:00:00"}
{"id_camion": "ERROR_SYSTEM", "velocidad": 0, "rpm": 0, "timestamp": "2023-01-01T13:00:00"}
{"id_camion": 106, "velocidad": 90, "rpm": 2100, "timestamp": "2023-01-01T13:00:00"
"""
# Nota: La última línea le falta la llave de cierre '}' a propósito.

dbutils.fs.put(f"{path_inbox}/sensor_data_batch_toxic.json", json_batch_toxic.strip(), overwrite=True)

print("Batch 04 (Tóxico) depositado. Esperando ingestión...")

In [0]:
# Rutas (Recuperamos las variables)
base_volume = "/Volumes/ecomotive/landing/ecomotive_vol"
path_inbox       = f"{base_volume}/inbox"
path_checkpoints = f"{base_volume}/sys/checkpoints/bronze_sensors"
path_schemas     = f"{base_volume}/sys/schemas/bronze_sensors"
target_table     = "ecomotive.landing.bronze_sensors"

# 1. READ STREAM
df_debug = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json") 
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data") 
  .option("cloudFiles.schemaLocation", path_schemas)
  .load(path_inbox)
)

# 2. WRITE STREAM
query_debug = (df_debug.writeStream
  .format("delta")
  .option("checkpointLocation", path_checkpoints)
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .table(target_table)
)

query_debug.awaitTermination()
print(f"Batch Tóxico procesado sin romper el pipeline.")

In [0]:
%sql
-- CONSULTA DE DEBUGGING: ¿Qué salió mal?
SELECT 
*
FROM ecomotive.landing.bronze_sensors 

In [0]:
%sql
-- CONSULTA DE DEBUGGING: ¿Qué salió mal?
SELECT * 
FROM ecomotive.landing.bronze_sensors 
WHERE _rescued_data IS NOT NULL;

In [0]:
%sql
-- CONSULTA DE DEBUGGING: ¿Qué salió mal?
SELECT 
  id_camion, 
  velocidad, 
  _rescued_data 
FROM ecomotive.landing.bronze_sensors 

In [0]:
%sql
DESCRIBE HISTORY ecomotive.landing.bronze_sensors;